# MSI edge2vec: local seed-0 reconstruction

Select a local **Python 3 kernel → Run All**. First use creates an independent Python 3.8 environment with mamba/conda and network access. The current kernel starts computation and displays results without switching kernels.

Pipeline: `MSI typed molecular graph → edge-type transition matrix / EM → biased walks → Gensim Skip-gram → Logistic Regression`.

Reuse the MSI graph and saved seed-0 negatives. Add no edges; use no MeSH/ATC or teleportation. New outputs go into `00.Reproduction/03.edge2vec_reproduction/`. Training starts when you run the cells; published outputs are cleared.


## 1. Directories and run name

Locate the `00.Reproduction/` layout automatically. If needed, set `REPOSITORY_OVERRIDE` to the repository root or reproduction directory. `PYTHON_OVERRIDE` may select an interpreter satisfying the pins. Use a new `RUN_NAME` after changing inputs, settings or implementation.


In [ ]:
from pathlib import Path
import csv
import html
import json
import os
import shutil
import signal
import subprocess
import sys
from IPython.display import display, HTML

RUN_NAME = 'msi_seed0'
REPOSITORY_OVERRIDE = None
PYTHON_OVERRIDE = None

def find_reproduction_root(start):
    start = Path(start).expanduser().resolve()
    for parent in [start, *start.parents]:
        for candidate in [parent / '00.Reproduction', parent]:
            if ((candidate / '00.DREAMwalk_reproduction').is_dir()
                    and (candidate / '03.edge2vec_reproduction').is_dir()):
                return candidate
    raise FileNotFoundError('00.Reproduction not found; set REPOSITORY_OVERRIDE to the repository root or 00.Reproduction.')

REPRODUCTION = find_reproduction_root(REPOSITORY_OVERRIDE or Path.cwd())
ROOT = REPRODUCTION / '03.edge2vec_reproduction'
SOURCE = REPRODUCTION / '00.DREAMwalk_reproduction'
ENV_PREFIX = ROOT / 'environment/.conda-env'
PYTHON = Path(PYTHON_OVERRIDE).expanduser().resolve() if PYTHON_OVERRIDE else ENV_PREFIX / 'bin/python'
RUNNER = ROOT / 'src/run_edge2vec.py'
if not RUN_NAME or any(c not in 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_-' for c in RUN_NAME):
    raise ValueError('RUN_NAME accepts only letters, digits, underscores and hyphens.')
RESULTS = ROOT / 'results' / RUN_NAME
LOGS = ROOT / 'logs' / RUN_NAME
LOGS.mkdir(parents=True, exist_ok=True)
for path in [SOURCE / 'data/processed/msi_graph.tsv',
             SOURCE / 'data/processed/msi_nodetypes.tsv',
             SOURCE / 'data/processed/msi_positive_pairs.tsv',
             SOURCE / 'results/msi_repeat0_negative_pairs.csv', RUNNER]:
    if not path.is_file():
        raise FileNotFoundError(f'Required input or runner missing: {path}')
print('Project: ', ROOT)
print('Shared inputs: ', SOURCE)
print('Computation interpreter: ', PYTHON)
print('Results: ', RESULTS)

## 2. Independent environment: installed on first run

Python 3.8; NumPy 1.23.3; SciPy 1.9.1; scikit-learn 1.1.3; NetworkX 2.8.8; pandas 1.4.4; Gensim 4.3.2; smart_open 6.4.0.

The environment lives at `environment/.conda-env/`; installation output goes to `logs/<RUN_NAME>/environment.log`. Reuse matching environments.


In [ ]:
def stream_command(command, log_path, env=None):
    command = list(map(str, command))
    print('Starting: ', ' '.join(command), flush=True)
    with log_path.open('a', encoding='utf-8') as log:
        log.write('\nCOMMAND: ' + repr(command) + '\n')
        with subprocess.Popen(command, cwd=ROOT, env=env,
                              stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, bufsize=1, start_new_session=True) as process:
            try:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    log.write(line)
                    log.flush()
                returncode = process.wait()
            except BaseException:
                try:
                    os.killpg(process.pid, signal.SIGTERM)
                except ProcessLookupError:
                    pass
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    try:
                        os.killpg(process.pid, signal.SIGKILL)
                    except ProcessLookupError:
                        pass
                    process.wait()
                raise
            if returncode:
                raise subprocess.CalledProcessError(returncode, command)

PINS = {'numpy': '1.23.3', 'scipy': '1.9.1', 'scikit-learn': '1.1.3',
        'networkx': '2.8.8', 'pandas': '1.4.4', 'gensim': '4.3.2',
        'smart_open': '6.4.0'}
probe = (
    'import sys; from importlib.metadata import version; '
    'import numpy, scipy, sklearn, networkx, pandas, gensim; '
    'assert sys.version_info[:2] == (3,8); '
    f'expected={PINS!r}; '
    'actual={k:version(k) for k in expected}; '
    'assert actual == expected, actual; print(sys.version); print(actual)'
)
ready = subprocess.run([str(PYTHON), '-c', probe], capture_output=True, text=True) if PYTHON.is_file() else None
if ready is None or ready.returncode:
    if PYTHON_OVERRIDE:
        if ready is not None:
            print(ready.stdout, ready.stderr)
        raise RuntimeError('PYTHON_OVERRIDE does not match package pins; clear it to install a dedicated environment.')
    candidates = [shutil.which('mamba'), shutil.which('conda'),
                  str(Path.home() / 'mambaforge/bin/mamba'),
                  str(Path.home() / 'miniforge3/bin/mamba'),
                  str(Path.home() / 'miniforge3/bin/conda'),
                  str(Path.home() / 'miniconda3/bin/conda')]
    manager = next((p for p in candidates if p and Path(p).is_file()), None)
    if manager is None:
        raise RuntimeError('mamba/conda not found; install Miniforge and rerun this section.')
    action = 'install' if (ENV_PREFIX / 'conda-meta/history').is_file() else 'create'
    command = [manager, action, '--yes', '--prefix', str(ENV_PREFIX),
               '--override-channels', '-c', 'conda-forge', 'python=3.8']
    command += [f'{package}={version}' for package, version in PINS.items()]
    install_env = dict(os.environ)
    install_env['CONDA_PKGS_DIRS'] = str(ROOT / 'environment/.conda-pkgs')
    stream_command(command, LOGS / 'environment.log', env=install_env)
    ready = subprocess.run([str(PYTHON), '-c', probe], capture_output=True, text=True)
    if ready.returncode:
        print(ready.stdout, ready.stderr)
        raise RuntimeError('Environment is not ready; inspect the installation log above.')
print(ready.stdout)

## 3. Fixed configuration

| Component | Setting |
|---|---|
| Graph | Shared typed MSI molecular graph; five edge types; no DDA treatment edges |
| Semantics / teleport | Neither is used; no MeSH/ATC mapping required |
| Source | Commit `b9250bd17112d203c2a5b496b75b4b6b259da9ed`; original snapshot and Python 3 adapter kept separately |
| EM matrix | Five iterations; one node walk/node per iteration; length 10; Pearson + sigmoid, metric 4 |
| Final walks | 100/node; length 10; p=q=1; 2,995,900 walks |
| Skip-gram | Standard Gensim; 128D; window 4; one epoch; negative=5; four workers; seed 0 |
| Prediction | Drug-minus-disease embeddings → Logistic Regression; L2; C=1; lbfgs; max_iter=1000; no scaling |
| Evaluation | Same 5,926 positives + 5,926 seed-0 negatives; StratifiedKFold(10, shuffle=True, random_state=0) |

This **edge2vec baseline** uses edge-type EM and biased walks, without DREAMwalk C++, `pp`, `τ` or XGBoost.

**Source adaptation:** preserve original files. The execution copy ports Python 2/Gensim APIs and corrects two issues: a Wilcoxon call overwriting metric-4 Pearson correlation, and assignment of the backtracking node to the wrong variable. A per-node sampling adapter converts node walks to edge-type tokens.

Five EM iterations follow upstream defaults. One epoch, subtraction and LR settings follow the node2vec comparator. The complete author MSI edge2vec benchmark script is unavailable. Record settings and inherited `PYTHONHASHSEED` without setting an extra startup value. Details are consolidated in the repository README.


In [ ]:
CHILD_ENV = dict(os.environ)
CHILD_ENV['PYTHONUNBUFFERED'] = '1'
CHILD_ENV['PYTHONDONTWRITEBYTECODE'] = '1'

def run_phase(phase):
    stream_command([PYTHON, '-u', RUNNER, phase, '--run-name', RUN_NAME],
                   LOGS / f'{phase}.log', env=CHILD_ENV)

run_phase('config')

## 4. Prepare typed edges and evaluation pairs

**Read MSI data and generate model inputs here.** Convert molecular edges into edge2vec format while retaining edge types. Treatment edges remain outside the graph; labeled pairs are used only downstream.

Save source versions, node/edge counts and SHA-256. Other model directories remain read-only.


In [ ]:
run_phase('prepare')

## 5. Learn the edge-type transition matrix

Run five EM/typed-walk iterations, generating one length-10 walk per node per iteration. Save the completed transition matrix for reuse. This learns edge-type bias; full Skip-gram walks are generated next.


In [ ]:
run_phase('transition')

## 6. Generate full biased walks

Generate 100 walks for each of 29,959 nodes: **2,995,900 walks**, length 10, p=q=1. Save the large `walks.txt` under `data/<RUN_NAME>/` and write its completion marker. Generated corpora are not part of this source release.


In [ ]:
run_phase('walks')

## 7. Train and immediately save embeddings

Standard Gensim Skip-gram: 128D, window 4, one epoch, negative=5 and four workers.

**Save `data/<RUN_NAME>/msi_edge2vec_seed0.pkl` before evaluation.** Reuse it after a later classifier/session interruption.


In [ ]:
run_phase('embedding')
print('Embedding saved: ', ROOT / 'data' / RUN_NAME / 'msi_edge2vec_seed0.pkl')

## 8. Ten-fold Logistic Regression

Reuse DREAMwalk's seed-0 negatives and node2vec's pair order, CV and LR settings: subtraction, L2, C=1, lbfgs, max_iter=1000 and no scaling.

Generate and save folds deterministically with the same seed. This stage does not retrain embeddings or alter graph/negatives. Accuracy uses threshold 0.5; AUPR is average precision.


In [ ]:
run_phase('evaluate')

## 9. Compare with the paper's edge2vec baseline

Use **Figure2a / edge2vec** Source Data, rather than DREAMwalk's score. Local evaluation is **one ten-fold repeat**, compared with **ten paper repeats**. No extra seeds, tuning or DREAMwalk module substitutions are started automatically.


In [ ]:
run_phase('compare')

def show_csv(path):
    with path.open(newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        columns = reader.fieldnames
        rows = list(reader)
    if not columns:
        print(path.name, ': empty table')
        return
    def pretty(value):
        try:
            return f'{float(value):.6f}'
        except (TypeError, ValueError):
            return str(value)
    header = ''.join(f'<th>{html.escape(c)}</th>' for c in columns)
    body = ''.join('<tr>' + ''.join(f'<td>{html.escape(pretty(row[c]))}</td>' for c in columns) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + header + '</tr></thead><tbody>' + body + '</tbody></table>'))

for filename in ['fold_results.csv', 'summary.csv', 'paper_comparison.csv']:
    print(filename)
    show_csv(RESULTS / filename)
print('Completed edge2vec seed-0 ten-fold evaluation.')
print('Results: ', RESULTS)
print('Embedding：', ROOT / 'data' / RUN_NAME / 'msi_edge2vec_seed0.pkl')

## Outputs and sources

- Transition matrix: `data/<RUN_NAME>/transition_matrix.txt`; corpus: `walks.txt`; node map: `nodes.json`.
- Embeddings: `data/<RUN_NAME>/msi_edge2vec_seed0.pkl`.
- Results: `fold_assignments.csv`, `fold_results.csv`, `oof_predictions.csv`, `summary.csv`, `paper_comparison.csv`.
- Configuration/completion JSON: `results/<RUN_NAME>/`; logs: `logs/<RUN_NAME>/`.
- Completed stages are reused. Use a new run name after source/input/parameter changes.
- Four-worker Skip-gram may vary numerically; saved configurations and artifacts identify each execution.

[Bang et al. (2023)](https://www.nature.com/articles/s41467-023-39301-y) · [Supplementary Method 1](https://media.springernature.com/original/springer-static/esm/art%3A10.1038%2Fs41467-023-39301-y/MediaObjects/41467_2023_39301_MOESM1_ESM.pdf) · [edge2vec](https://github.com/RoyZhengGao/edge2vec) · [Pinned source](https://github.com/RoyZhengGao/edge2vec/tree/b9250bd17112d203c2a5b496b75b4b6b259da9ed)
